In [1]:
import pandas as pd

In [2]:
with open('/kaggle/input/complete-works-of-rabindranath-tagore/txt/poem.txt','r',) as file:
    rabindranath = file.read()

In [3]:
rabindranath[:100]

'বজাও রে মোহন বাঁশি।\nসারা দিবসক\nবিরহদহনদুখ,\nমরমক তিয়াষ নাশি।\nরিঝমনভেদন\nবাঁশরিবাদন\nকঁহা শিখলি রে কান?\n'

In [4]:
len(rabindranath)

2368921

In [5]:
with open('/kaggle/input/complete-works-of-kazi-nazrul-islam/txt/poem.txt','r',) as file:
    kazi_nazrul = file.read()

In [6]:
def split_into_chunks(text, N):
    chunk_size = len(text) // N
    chunks = []
    for i in range(N):
        start = i * chunk_size
        # For the last chunk, take all remaining text to handle uneven division
        if i == N - 1:
            end = len(text)
        else:
            end = (i + 1) * chunk_size
        chunks.append(text[start:end])
    return chunks


# Number of chunks (5 chunks = 20% each)
N = 50

# Split each text into 5 chunks
chunks_a = split_into_chunks(rabindranath, N)
chunks_b = split_into_chunks(kazi_nazrul, N)

# Merge by interleaving chunks: a1 + b1 + a2 + b2 + ... + a10 + b10
text = ''.join(a + b for a, b in zip(chunks_a, chunks_b))

In [7]:
len(text)

3146413

In [8]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import os

# hyperparameters
batch_size = 128 # how many independent sequences will we process in parallel?
block_size = 256  # what is the maximum context length for predictions?
max_iters = 10000
eval_interval = 500
learning_rate = 4e-4 # self attention can't take high learning rate.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embed = 384 # so head_size = 384/6 = 64 dimension
n_head = 6
n_layer = 6
dropout = 0.2
# ------------

torch.manual_seed(555)

#_________Preprocessing__________

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

#__________________________

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) 
train_data = data[:n]
val_data = data[n:]
#__________________________

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()
        self.query = nn.Linear(n_embed,head_size,bias=False)
        self.key = nn.Linear(n_embed,head_size,bias=False)
        self.value = nn.Linear(n_embed,head_size,bias=False)
        self.register_buffer('tril',torch.tril(torch.ones(block_size,block_size)))
        self.dropout = nn.Dropout(dropout)
    
    def forward(self,x):
        B,T,C = x.shape
        q = self.query(x)
        k = self.key(x)
        weight = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T) Scaled dot product attention 
        weight= weight.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T) lower triangular matrix
        weight= F.softmax(weight, dim=-1) # (B, T, T)
        weight = self.dropout(weight)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = weight @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out
        
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.projection = nn.Linear(n_embed, n_embed)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(out)
        out = self.projection(out)
        return out

class FFN(nn.Module):

    def __init__(self, n_embed):
        super().__init__()
        self.nn = nn.Sequential(
            nn.Linear(n_embed,n_embed*4),# multiplied by 4 because mentioned in paper
            nn.ReLU(),
            nn.Linear(4*n_embed,n_embed),
            nn.Dropout(dropout)
        )
    def forward(self, x):
        out = self.nn(x)
        return out

class Block(nn.Module):

    def __init__(self, n_embed,n_head):
        super().__init__()
        head_size = n_embed//n_head
        self.self_atte_heads = MultiHeadAttention(n_head, head_size) # communication 
        self.ffn = FFN(n_embed) # computation
        self.l1 = nn.LayerNorm(n_embed)
        self.l2 = nn.LayerNorm(n_embed)

    def forward(self,x):
        #ressidual connection 
        x = self.l1(x)
        x = x + self.self_atte_heads(x)
        x = self.l2(x)
        x = x + self.ffn(x)
        return x
        
    
class LLM(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.apply(self._init_weights)
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.positional_embedding_table = nn.Embedding(block_size,n_embed)
        self.blocks = nn.Sequential(*[Block(n_embed,n_head) for i in range(n_layer)])
        self.layernorm = nn.LayerNorm(n_embed)
        self.linear_head = nn.Linear(n_embed, vocab_size)
        

    def _init_weights(self,module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, idx, targets=None):

        B,T = idx.shape
        # idx and targets are both (B,T) tensor of integers
        token_embed = self.token_embedding_table(idx) # (B,T,C)
        pos_embed = self.positional_embedding_table(torch.arange(T,device=device)) # T,C
        x = token_embed + pos_embed
        x = self.blocks(x)
        logits = self.linear_head(x)

        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last_block_size tokens
            idx_cond = idx[:,-block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

# model = LLM()
# m = model.to(device)

# # create a PyTorch optimizer
# optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# checkpoint_path = '/kaggle/working/best_model.pth'
# if os.path.exists(checkpoint_path):
#     print(f"Loading checkpoint from {checkpoint_path}")
#     checkpoint = torch.load(checkpoint_path)
#     model.load_state_dict(checkpoint['model_state_dict'])
#     optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
#     start_iter = checkpoint['iteration']
#     best_val_loss = checkpoint['loss']
#     print(f"Resuming from iteration {start_iter} with best val loss: {best_val_loss:.4f}")
    
# for iter in range(max_iters):

#     # every once in a while evaluate the loss on train and val sets
#     if iter % eval_interval == 0:
#         losses = estimate_loss()
#         print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")


    
#     if losses['val'] < losses['train']:
#         best_loss = losses['val']
#         best_epoch = max_iters
#         # Save the best model
#         torch.save({
#             'iteration': iter,
#             'epoch': losses['val'],
#             'model_state_dict': model.state_dict(),
#             'optimizer_state_dict': optimizer.state_dict(),
#             'loss': losses['val'],
#         },checkpoint_path)

#     # training
#     # sample a batch of data
#     xb, yb = get_batch('train')

#     # evaluate the loss
#     logits, loss = model(xb, yb)
#     optimizer.zero_grad(set_to_none=True)
#     loss.backward()
#     optimizer.step()

# generate from the model
# context = torch.zeros((1, 1), dtype=torch.long, device=device)
# print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))

In [9]:
model = LLM()
m = model.to(device)

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
checkpoint_path = '/kaggle/working/checkpoint1.pth'

In [10]:
print(model)

LLM(
  (token_embedding_table): Embedding(144, 384)
  (positional_embedding_table): Embedding(256, 384)
  (blocks): Sequential(
    (0): Block(
      (self_atte_heads): MultiHeadAttention(
        (heads): ModuleList(
          (0-5): 6 x Head(
            (query): Linear(in_features=384, out_features=64, bias=False)
            (key): Linear(in_features=384, out_features=64, bias=False)
            (value): Linear(in_features=384, out_features=64, bias=False)
            (dropout): Dropout(p=0.2, inplace=False)
          )
        )
        (projection): Linear(in_features=384, out_features=384, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
      )
      (ffn): FFN(
        (nn): Sequential(
          (0): Linear(in_features=384, out_features=1536, bias=True)
          (1): ReLU()
          (2): Linear(in_features=1536, out_features=384, bias=True)
          (3): Dropout(p=0.2, inplace=False)
        )
      )
      (l1): LayerNorm((384,), eps=1e-05, elementwise_affine=T

In [11]:
from torchinfo import summary
sample_input = torch.randint(0, vocab_size, (1, block_size), dtype=torch.long, device=device)
summary(m, input_data=sample_input)

Layer (type:depth-idx)                        Output Shape              Param #
LLM                                           [1, 256, 144]             768
├─Embedding: 1-1                              [1, 256, 384]             55,296
├─Embedding: 1-2                              [256, 384]                98,304
├─Sequential: 1-3                             [1, 256, 384]             --
│    └─Block: 2-1                             [1, 256, 384]             --
│    │    └─LayerNorm: 3-1                    [1, 256, 384]             768
│    │    └─MultiHeadAttention: 3-2           [1, 256, 384]             590,208
│    │    └─LayerNorm: 3-3                    [1, 256, 384]             768
│    │    └─FFN: 3-4                          [1, 256, 384]             1,181,568
│    └─Block: 2-2                             [1, 256, 384]             --
│    │    └─LayerNorm: 3-5                    [1, 256, 384]             768
│    │    └─MultiHeadAttention: 3-6           [1, 256, 384]            

In [12]:
import wandb
wandb.login(key="REDACTED_WANDB_API_KEY")


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: tahmidulkashfi15 (tahmidulkashfi15-international-islamic-university-chittagong). Use `wandb login --relogin` to force relogin
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [13]:
resume_from = False

In [14]:


wandb.init(
    project="BanglaGPT1", 
    config={
        "batch_size": batch_size,
        "block_size": block_size,
        "max_iters": max_iters,
        "eval_interval": eval_interval,
        "learning_rate": learning_rate,
        "eval_iters": eval_iters,
        "n_embd": n_embed,
        "n_head": n_head,
        "n_layer": n_layer,
        "dropout": dropout,
    },
    notes=f"Resumed from {resume_from}" if resume_from else None
)
# Initialize training variables
start_iter = 0
best_loss = float('inf')
best_epoch = 0  # In your code, this is actually the best iteration

# Load checkpoint if resuming
if resume_from:
    checkpoint = torch.load(resume_from)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_iter = checkpoint['iteration'] + 1
    best_loss = checkpoint['loss']
    best_epoch = checkpoint['epoch']
    print(f"Resumed from iteration {checkpoint['iteration']}, best val loss {best_loss}")

for iter in range(max_iters):
    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        
        # Log metrics to wandb
        wandb.log({
            "train_loss": losses['train'],
            "val_loss": losses['val'],
            "iteration": iter
        })

    if losses['val'] < best_loss:  # Changed condition to compare with best_loss
        best_loss = losses['val']
        best_epoch = iter
        
        # Save the best model using wandb
        checkpoint = {
            'iteration': iter,
            'epoch': best_epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': best_loss,
        }
        
        # Save locally
        torch.save(checkpoint, checkpoint_path)
        
        # Save to wandb
        wandb.save(checkpoint_path)
        
        # Log best model metrics
        wandb.run.summary["best_val_loss"] = best_loss
        wandb.run.summary["best_epoch"] = best_epoch

    # training
    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
    # Log training progress
    if iter % 500 == 0:  # Log every 500 iterations
        wandb.log({
            "training_loss": loss.item(),
            "learning_rate": optimizer.param_groups[0]['lr'],
            "iteration": iter
        })

# # Finish the wandb run
wandb.finish()

wandb: WARNING Saving files without folders. If you want to preserve subdirectories pass base_path to wandb.save, i.e. wandb.save("/mnt/folder/file.h5", base_path="/mnt")


step 0: train loss 5.0981, val loss 5.1017
step 500: train loss 2.3152, val loss 2.3498
step 1000: train loss 1.9169, val loss 1.9696
step 1500: train loss 1.7625, val loss 1.8314
step 2000: train loss 1.6704, val loss 1.7711
step 2500: train loss 1.6100, val loss 1.7343
step 3000: train loss 1.5574, val loss 1.7140
step 3500: train loss 1.5194, val loss 1.7025
step 4000: train loss 1.4782, val loss 1.6931
step 4500: train loss 1.4454, val loss 1.6895
step 5000: train loss 1.4108, val loss 1.6883
step 5500: train loss 1.3803, val loss 1.6887
step 6000: train loss 1.3515, val loss 1.6966
step 6500: train loss 1.3270, val loss 1.7036
step 7000: train loss 1.2982, val loss 1.6928


KeyboardInterrupt: 

In [29]:
print(text[300000:301000])

 যে পথে কঙ্কাল পায়ে বাজে!
নির্যাতনের যে যষ্টি দিয়া শত্রু আঘাত হানে
সেই যষ্টিরে দোসর করিয়া অলক্ষ্য পথ-পানে
চলেছে দেবতা – অন্ধ দেবতা – পায়ে পায়ে পলে পলে,
যত ঘিরে আসে পথ-সংকট চলে তত নববলে।
ঢলে পড়ে পথ পরে,
নবীন মৃত্যু-যাত্রী আসিয়া তুলে ধরে বুকে করে!
অন্ধ কারার বন্ধ দুয়ারে যথায় বন্দি জাগে,
যথায় বধ্য-মঞ্চ নিত্য রাঙিছে রক্ত-রাগে,
যথায় পিষ্ট হতেছে আত্মা নিষ্ঠুর মুঠি-তলে,
যথায় অন্ধ গুহায় ফণীর মাথায় মানিক জ্বলে,সন্ধ্যা
যথায় বন্য শ্বাপদের সাথে নখর দন্ত লয়ে
জাগে বিনিদ্র বন্য-তরুণ ক্ষুধার তাড়না সয়ে,
যথা প্রাণ দেয় বলির নারীরা যূপকাষ্ঠের ফাঁদে, −
সেই পথে চলে অন্ধ দেবতা, পথ চলে আর কাঁদে, −
‘ওরে ওঠ ত্বরা করি
তোদের রক্তে-রাঙা উষা আসে, পোহাইছে বিভাবরী!’
তিমির রাত্রি, ছুটেছে যাত্রী নিরুদ্দেশের ডাকে,
জানে না কাথায় কোন পথে কোন ঊর্ধ্বে দেবতা হাঁকে।
শুনিয়াছে ডাক এই শুধু জানে! অপনার অনুরাগে
মাতিয়া উঠেছে অলস চরণ, সম্মুখে পথ জাগে!
জাগে পথ, জাগে ঊর্ধ্বে দেবতা, এই দেখিয়াছে শুধু,
কে দেখে সে পথে চোরা বালুচর, পর্বত, মরু ধুধু!
ছুটেছে পথিক, সাথে চলে পথ, অমানিশি চলে সাথে,
পথে পড়ে ঢলে, মৃত্যুর ছলে ধরে দেবতার হাতে।
চলিতেছ

In [20]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=1000)[0].tolist()))


প্রথম এসেছিল ভেবে, সে আর আশা-মথি বুথি লাগি হাত হাড়ি,
কেন সে পথ চিনা ঘুরবে করিতে বৃথাই কারা পাতা পাতা,
সেই মহামণি হইল বক্ষ-ক্ষে ধুলাতুর সাথে আসে,
চাহিয়া ছাড়ি গো হেরি নিশ্চল বাতা।
চাইনি চাহি চাহিতে —
পারি না কেহ চিরকাল নীরব বসিতে লাগিতে চারি
পাড়নির এই গহনজাগর গোপন নিশ্বাসে,
নিমেষে চেয়ে আছে অনন্ত চরণ কাঁদিনী কাঁদিয়া উঠিতে চাহিতে
আমার কি কুহের মুখের ভরে।
তার পরে দীপ হইতে শুধু জাগিতেছিল চাঁদিয়া,
ভেবে গেলেম পুব হাসিতেছিল চাহিয়া, ছুটে গেল বুকের পুরায়ে,
কেননার আশীর্বাদে পুরাতে পুড়িল খুঁজিয়া এসে ধুলার পিছু
সহসা উষার ইনবরাত হইয়া এসেছে কোণে কেহ কোথা সুথার করিল মায়ে।
হেরিনু চবতোমা হতে ধূমকেতু মাহিলের জমজাতের প্রভাত বাতাস,
হে পথিক
অন্তরযুগের মৃতদেহের মরমেয়ে, চন্দ্রালোকে মরু মমরচে-মরু-ভূমি
রুদ্র অভিসারে আনিল ফিরে। জহরত সাধুসিংহীন মরুর জেনে না আর মুগ্ধ ঘটে যায়।
ভোগ্য-বল্প ভাবে, ভুলে যদি আরোগ্য আজ যাতে, মরুতে কুল-কাদা ফেরতা বেরো বলতে অবমানবের আর,
ইসলাম তাহারই দন্তী করে মেনে না এমনই লজ্জারে ডাকবে হরু!
শুনেছিল তারই স্বর্গ-যারি, জিনি নিজে যার আঁধার ঘুমায়ে উঠিল বুঝিয়া না ভেবে,
যাহাদের এ-মক্কার অপূর্ব নির